# Lesson 02 · Functors

We just built categories. The reflex of the whole subject is to immediately ask for the right maps between them — and answering snaps categories themselves into a category.

This notebook collects every Python block from the optional labs and exercises of [the lesson page](../lesson-02-functors.html), in page order. Run the cells from top to bottom; later cells use names defined in earlier ones.

Python is optional in this course — the lesson is complete without it. *Generated by `tools/sync_code.py`: edit the lesson page, not this file.*

## §7 — Functors by hand

**`list_functor.py`** — map is a functor

In [1]:
def fmap(f, xs):
    "lift an ordinary function to act on a list, elementwise"
    return [f(x) for x in xs]

inc = lambda x: x + 1
sq  = lambda x: x * x
data = [1, 2, 3, 4]

# Functor law 1:  fmap(id) == id
assert fmap(lambda x: x, data) == data

# Functor law 2:  fmap(g . f) == fmap(g) . fmap(f)
assert fmap(lambda x: sq(inc(x)), data) == fmap(sq, fmap(inc, data))

print(fmap(inc, data))            # [2, 3, 4, 5]

[2, 3, 4, 5]


**`setup.py`** — Lesson 1’s category checker, carried over

In [2]:
from itertools import product

class FiniteCategory:
    """objects, arrows (name -> (dom, cod)), a composition table,
       and a chosen identity arrow for each object."""
    def __init__(self, objects, arrows, comp, ident):
        self.obj, self.ar = set(objects), arrows
        self.comp, self.id = comp, ident

    def dom(self, a): return self.ar[a][0]
    def cod(self, a): return self.ar[a][1]
    def composable(self, g, f): return self.dom(g) == self.cod(f)

    def check(self):
        # 1. each identity is typed X -> X
        for X in self.obj:
            assert self.ar[self.id[X]] == (X, X)
        # 2. every composable pair has a composite, correctly typed
        for g, f in product(self.ar, repeat=2):
            if self.composable(g, f):
                gf = self.comp[(g, f)]
                assert self.ar[gf] == (self.dom(f), self.cod(g))
        # 3. identity law on both sides
        for f in self.ar:
            X, Y = self.ar[f]
            assert self.comp[(self.id[Y], f)] == f
            assert self.comp[(f, self.id[X])] == f
        # 4. associativity
        for h, g, f in product(self.ar, repeat=3):
            if self.composable(h, g) and self.composable(g, f):
                assert self.comp[(h, self.comp[(g, f)])] == \
                       self.comp[(self.comp[(h, g)], f)]
        return True

**`functor.py`** — a functor between finite categories

In [3]:
class Functor:
    """F : C -> D, given by a map on objects and a map on arrows."""
    def __init__(self, C, D, on_obj, on_arr):
        self.C, self.D, self.on_obj, self.on_arr = C, D, on_obj, on_arr

    def check(self):
        C, D, Fo, Fa = self.C, self.D, self.on_obj, self.on_arr
        for f in C.ar:                          # typing: F(f): F(dom) -> F(cod)
            X, Y = C.ar[f]
            assert D.ar[Fa[f]] == (Fo[X], Fo[Y])
        for X in C.obj:                         # preserves identities
            assert Fa[C.id[X]] == D.id[Fo[X]]
        for g, f in product(C.ar, repeat=2):    # preserves composition
            if C.composable(g, f):
                assert Fa[C.comp[(g, f)]] == D.comp[(Fa[g], Fa[f])]
        return True

# (Z/3, +) as a one-object category, carried over from Lesson 1
ar   = {n: ('*', '*') for n in range(3)}
comp = {(a, b): (a + b) % 3 for a in range(3) for b in range(3)}
Z3 = FiniteCategory({'*'}, ar, comp, {'*': 0})

# "multiply by 2" is a monoid homomorphism -- i.e. a functor of one-object cats
double = Functor(Z3, Z3, on_obj={'*': '*'},
                 on_arr={n: (2 * n) % 3 for n in range(3)})
print(double.check())

True


## Exercise 15

**`two_to_z3.py`**

In [4]:
# the category 2 = ( 0 --u--> 1 )
two_ar   = {'i0':(0,0), 'i1':(1,1), 'u':(0,1)}
two_comp = {('i0','i0'):'i0', ('i1','i1'):'i1',
            ('u','i0'):'u',  ('i1','u'):'u'}
Two = FiniteCategory({0,1}, two_ar, two_comp, {0:'i0', 1:'i1'})

for pick in (0, 1, 2):
    F = Functor(Two, Z3, on_obj={0:'*', 1:'*'},
                on_arr={'i0':0, 'i1':0, 'u':pick})
    print(pick, F.check())        # all three pass

0 True
1 True
2 True


## Exercise 16

**`shift.py`** — a map that is not a functor

In [5]:
shift = Functor(Z3, Z3, on_obj={'*': '*'},
                on_arr={n: (n + 1) % 3 for n in range(3)})
try:
    shift.check()
except AssertionError:
    print("rejected: the identity 0 is sent to 1")

rejected: the identity 0 is sent to 1


## Exercise 18

**`powerset.py`**

In [6]:
from itertools import combinations

def pmap(f, S):                       # direct image on subsets
    return {f(x) for x in S}

def subsets(S):
    S = list(S)
    return [set(c) for r in range(len(S)+1) for c in combinations(S, r)]

h1 = lambda x: x % 2
h2 = lambda x: x + 1
for S in subsets({1, 2, 3}):
    assert pmap(lambda x: x, S) == S
    assert pmap(h1, pmap(h2, S)) == pmap(lambda x: h1(h2(x)), S)
print("powerset functor laws: 16 instances pass")

powerset functor laws: 16 instances pass
